### Программистская часть

#### Задача 1.

Напишите черновик игры в жанре RPG. Идеологически: игрок выбирает, будет ли он играть за волшебника или за бойца, а потом выбранным героем сражается с монстрами, набирая очки опыта. Что должно быть технически:

- классы волшебника и бойца (можно создать отдельный класс Player и наследоваться от него, но необязательно)
- класс монстра (хотя бы один)
- класс оружия (тут тоже фантазию не ограничиваю - можно создать абстрактный класс и наследоваться от него, можно сделать классы для меча и для посоха с варьирующими атрибутами)
- класс Игра, в котором будут все необходимые методы
- все это должно быть разложено по отдельным скриптам .py в папке, класс игры импортируется в файл main.py, и его методы вызываются там.

In [ ]:
# код в отдельных файлах RPG

#### Задача 2.

Дан текст, каждая строка которого является полным или относительным путём к некоторому файлу.
Напишите регулярное выражение, которое захватывает:
1. директорию, в которой лежит файл;
2. только имя файла (без расширения);
3. только расширение;
При этом:
- нужны только файлы, у которых расширение не .bat и не .txt.
- пути могут быть как в Unix, так и в Windows формате (https://ru.wikipedia.org/wiki/Путь_к_файлу).
- расширение, если оно есть, начинается с точки. Файлы могут быть без расширения вовсе (в этом случае на месте расширения должно стоять None или "")
- скрытые файлы могут начинаться с точки (например, .bashrc - и это не расширение)
- относительный путь может содержать только название файла, в этом случае вместо директории выведите None или ""
- в остальных случаях директория должна заканчиаться на разделитель директорий. Наприемр, в Unix-системах - "/" - это путь к корневой директории.
Требуется получить список из кортежей, каждый кортеж содержит извлечённые данные.
Используйте флаг VERBOSE, чтобы не запутаться.
(Расширение в целом может содержать всё, что угодно, но разделителей директорий не может быть в именах файлах и расширениях. https://en.wikipedia.org/wiki/List_of_filename_extensions )

In [ ]:
import re

def process_paths(paths):
    pattern = re.compile(r'(?m)' +
                         r'^([\\/]?(?:[^\\/:*?\"<>|\r\n]+[\\/])*)' +
                         r'(\.?(?:[^\\/:*?\"<>|\r\n\.]|\.(?=.*\.))+)' +
                         r'(\.[^\\/:*?\"<>|\r\n\.]+)?$')
    return re.findall(pattern, paths)

paths = '''
file.txts
\\dir\\file.bats
/dir/dir/file.py
dir\\file.py
dir\\file.before.py
.file
/dir/.file
file.py
/file.py
/file.py/file.py
dir1/file 1.7z
file
'''

expected_results = [
    ('', 'file', '.txts'),
    ('\\dir\\', 'file', '.bats'),
    ('/dir/dir/', 'file', '.py'),
    ('dir\\', 'file', '.py'),
    ('dir\\', 'file.before', '.py'),
    ('', '.file', ''),
    ('/dir/', '.file', ''),
    ('', 'file', '.py'),
    ('/', 'file', '.py'),
    ('/file.py/', 'file', '.py'),
    ('dir1/', 'file 1', '.7z'),
    ('', 'file', '')
]

actual_results = process_paths(paths)

assert(actual_results == expected_results)

#### Задача 3.

Жизнь.
Напишите игру "Жизнь".
Что это такое - читайте в википедии и здесь: http://www.michurin.net/online-tools/life-game.html
Вообще говоря, это не игра в привычном понимании этого слова, а процесс.
В простейшем виде достаточно раз в 0.1 секунды выводить на экран обновлённое поле. Для рамочек можно использовать специальные символы для рисования рамочек (найдите в таблице unicode). Пробел - пустая клетка, живая клетка может быть обозначена, например, символом '+'. Начальное поле генерируется случайным образом, вероятность появления жизни в клетке при начальной генерации - должна быть настраиваемым параметром. Размеры поля вводит пользователь при запуске программы. Также должна быть возможность в качестве начальной популяции использовать R-pentomino (http://www.conwaylife.com/wiki/R-pentomino)

In [ ]:
import numpy as np
import time

from IPython.display import clear_output

class Life:
    def __init__(self, width=60, height=40, prob=0.3, time=0.1):
        self._prob = prob
        self._width = width
        self._height = height
        self._time = time

    def _initialize_field(self, randomly=True):
        if randomly:
            self._field = np.random.random((self._height, self._width)) < self._prob
        else:
            self._field = np.full((self._height, self._width), False)
            cx = self._width // 2
            cy = self._height // 2
            self._field[cy - 1, cx] = True
            self._field[cy - 1, cx + 1] = True
            self._field[cy, cx] = True
            self._field[cy, cx - 1] = True
            self._field[cy + 1, cx] = True

        # массив для подсчёта живых соседей с "рамкой"
        self._scores = np.zeros((self._height + 2, self._width + 2))

    def _one_step(self):
        self._scores.fill(0)
        # сместим field 8 раза в разных направлениях и прибавим к scores
        for i in range(3):
            for j in range(3):
                if i == 1 and j == 1:
                    continue

                self._scores[i: i + self._height, j: j + self._width] += self._field

        # мёртвые клетки с 3 живыми соседями
        condition1 = (~self._field) & (self._scores[1: self._height + 1, 1: self._width + 1] == 3)
        # живые клетки с 2 или 3 живими соседями
        condition2 = self._field & ((self._scores[1: self._height + 1, 1: self._width + 1] == 2) | (self._scores[1: self._height + 1, 1: self._width + 1] == 3))

        self._field = condition1 | condition2


    def _show_field(self):
        horizontal_line = u'\u2501'
        vertical_line = u'\u2503'
        top_left_corner = u'\u250F'
        top_right_corner = u'\u2513'
        bottom_left_corner = u'\u2517'
        bottom_right_corner = u'\u251B'

        result = top_left_corner + horizontal_line * self._width + top_right_corner + '\n'
        for i in range(self._height):
            result += vertical_line + ''.join('+' if x else ' ' for x in self._field[i]) + vertical_line + '\n'
        result += bottom_left_corner + horizontal_line * self._width + bottom_right_corner
        clear_output(wait=True)
        print(result)


    def run(self, randomly=True):
        self._initialize_field(randomly)
        while True:
            self._show_field()
            time.sleep(self._time)
            self._one_step()


In [ ]:
print("Введите через пробел ширину и высоту поля")
size = input().split()
width, height = int(size[0]), int(size[1])
print("Введите R, если хотите использовать в качестве начальной популяции R-pentomino (иначе любое другое значение)")
randomly = input() != "R"
if randomly:
    print("Введите вероятность появления жизни")
    prob = float(input())
    life = Life(width, height, prob)
else:
    life = Life(width, height)
life.run(randomly)


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃                                        ┃
┃                                        ┃
┃                                        ┃
┃                                        ┃
┃                                        ┃
┃                                        ┃
┃                                        ┃
┃                                        ┃
┃                    ++                  ┃
┃                   +  +                 ┃
┃         ++         ++                  ┃
┃         ++                             ┃
┃                                        ┃
┃                                        ┃
┃                                        ┃
┃                                        ┃
┃                                        ┃
┃                                        ┃
┃                                        ┃
┃                                        ┃
┃      ++               ++             ++┃
┃      ++               ++             ++┃
┃          

### Лингвистическая часть

Для выполнения этих заданий выберите два любых достаточно длинных текста (.txt) на русском и на любом другом (для которого есть парсеры) языке; если возьмете текст и его перевод, будет отлично.

#### Задача 4.

Просмотрите оба выбранных текста. Удостоверьтесь, что тексты чистые, если же в них есть какой-то мусор: хештеги, затесавшиеся при OCR символы и подобное, почистите с помощью регулярных выражений.

Проведите первичный статистический анализ: разбейте тексты на предложения и на токены, посчитайте относительное количество того и другого, сопоставьте. Если ваши тексты параллельные, какой длиннее? В каком тексте средняя длина предложения больше? Почему? В каком тексте выше лексическое разнообразие?

Таким образом, вам необходимо узнать следующие вещи:

- количество предложений (относительное и абсолютное)
- количество токенов (относительное и абсолютное)
- средняя длина предложения (среднее количество слов в предложении)
- соотношение "уникальные токены / все токены"
- (опционально) соотношение знаков пунктуации и слов

In [ ]:
!pip install nltk
import nltk
nltk.download('punkt')
import re

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


In [ ]:
def preprocess_text(filepath, regex):
    with open(filepath, 'r') as file:
        text = file.read()
        text = re.sub(regex, '', text)
        text = re.sub(r'\s+', ' ', text)

        return text


In [ ]:
from tabulate import tabulate
from nltk.tokenize import sent_tokenize, word_tokenize

def analyze_text(text, word_regex):
    sents = sent_tokenize(text)
    tokens = word_tokenize(text)

    reg = re.compile(word_regex)
    words = [token for token in tokens if re.fullmatch(reg, token)]

    sent_lengths = [len(word_tokenize(sent)) for sent in sents]
    mean_length = sum(sent_lengths) / len(sent_lengths)

    results = {
        'кол-во предложений': len(sents),
        'кол-во токенов': len(tokens),
        'средняя длина предложения': mean_length,
        'уникальные токены/все токены': len(set(tokens)) / len(tokens),
        'знаки пунктуации/слова': (len(tokens) - len(words)) / len(words)
    }

    return results


text_ru = preprocess_text('prestuplenie-i-nakazanie.txt', r'[^А-Яа-яЁё.,:;!?–—\-()"\s]+')
text_de = preprocess_text('verbrechen_und_strafe.txt', r'[^A-Za-zÄäÖöÜüß.,:;!?–—\-()"\s]+')

analysis_ru = analyze_text(text_ru, r'[А-Яа-яЁё\-]+')
analysis_de = analyze_text(text_de, r'[A-Za-zÄäÖöÜüß]+')

table_data = [
    ['', 'ru', 'de', 'ru/de'],
]

for key in analysis_ru.keys():
    table_data.append([key, analysis_ru[key], analysis_de[key], analysis_ru[key] / analysis_de[key]])

print(tabulate(table_data, headers="firstrow", tablefmt="grid"))

+------------------------------+---------------+----------------+----------+
|                              |            ru |             de |    ru/de |
+==============================+===============+================+==========+
| кол-во предложений           |  13690        |  14307         | 0.956874 |
+------------------------------+---------------+----------------+----------+
| кол-во токенов               | 218838        | 246203         | 0.888852 |
+------------------------------+---------------+----------------+----------+
| средняя длина предложения    |     15.9866   |     17.2086    | 0.928993 |
+------------------------------+---------------+----------------+----------+
| уникальные токены/все токены |      0.126614 |      0.0682079 | 1.8563   |
+------------------------------+---------------+----------------+----------+
| знаки пунктуации/слова       |      0.284751 |      0.193405  | 1.4723   |
+------------------------------+---------------+----------------+----------+

#### Задача 5.

Сделайте морфосинтаксические разборы ваших текстов в формате UD, запишите .conllu-файлы.

In [ ]:
! pip install spacy_udpipe

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.0/937.0 kB 15.7 MB/s eta 0:00:00


In [ ]:
! pip install spacy-conll

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import spacy_udpipe
import spacy_conll
spacy_udpipe.download('ru')

Downloaded pre-trained UDPipe model for 'ru' language


In [ ]:
spacy_udpipe.download('de')

Downloaded pre-trained UDPipe model for 'de' language


In [ ]:
def make_ud(langcode, text):
  nlp = spacy_udpipe.load(langcode)
  nlp.max_length = 1500000
  nlp.add_pipe("conll_formatter", last=True)

  doc = nlp(text)
  with open(f"{langcode}.conllu", 'w') as f:
    f.write(doc._.conll_str)

  #  буду потом использовать
  return doc._.conll_pd

In [ ]:
nlp = spacy_udpipe.load('ru')
nlp.max_length = 1500000 # random large number above the limit
nlp.add_pipe("conll_formatter", last=True)


ConllFormatter(conversion_maps=None, ext_names={'conll_str': 'conll_str', 'conll': 'conll', 'conll_pd': 'conll_pd'}, field_names={'ID': 'ID', 'FORM': 'FORM', 'LEMMA': 'LEMMA', 'UPOS': 'UPOS', 'XPOS': 'XPOS', 'FEATS': 'FEATS', 'HEAD': 'HEAD', 'DEPREL': 'DEPREL', 'DEPS': 'DEPS', 'MISC': 'MISC'}, include_headers=False, disable_pandas=False)

In [ ]:
# текст большой, поэтому 5 минут занимает
doc = nlp(text_ru)

In [ ]:
doc_de = make_ud('de',text_de)

In [ ]:
doc_de

,ID,FORM,LEMMA,UPOS,XPOS,FEATS,HEAD,DEPREL,DEPS,MISC
0,1,ERSTER,erst,PROPN,ADJA,Case=Nom|Gender=Masc|Number=Sing,3,dep,_,_
1,2,BAND,Band,PROPN,NN,Case=Nom|Gender=Masc|Number=Sing,1,flat,_,_
2,3,Erster,erst,PROPN,ADJA,Case=Nom|Gender=Masc|Number=Sing,17,nsubj,_,_
3,4,Teil,Teil,PROPN,NN,Case=Nom|Gender=Masc|Number=Sing,3,flat,_,_
4,5,I,I,PROPN,FM,Foreign=Yes,3,appos,_,_
...,...,...,...,...,...,...,...,...,...,...
248360,17,Ende,Ende,NOUN,NN,Case=Acc|Gender=Neut|Number=Sing,15,obl,_,SpaceAfter=No
248361,18,.,.,PUNCT,$.,_,9,punct,_,_
248362,19,-,-,PUNCT,$(,_,20,punct,_,_
248363,20,ENDE,Ende,PROPN,NN,Case=Dat|Gender=Neut|Number=Sing,9,compound,_,_


In [ ]:
doc_ru = make_ud('ru',text_ru)

#### Задача 6.

Посчитайте статистику по частям речи, сопоставьте: можно напечатать две таблички с процентами по частям речи.

In [ ]:
import pandas

In [ ]:
def pos_stats(df):

  counts = df['UPOS'].value_counts()
  stats = pandas.DataFrame()
  stats.index = counts.index
  stats['count'] = counts.values
  stats['percent'] = stats['count']/stats['count'].sum() * 100

  return stats

In [ ]:
pos_stats(doc_de)

,count,percent
DET,3,20.000000
NOUN,3,20.000000
PRON,2,13.333333
VERB,2,13.333333
PUNCT,2,13.333333
ADP,2,13.333333
PART,1,6.666667


In [ ]:
pos_stats(doc_ru)

,count,percent
PUNCT,49862,22.539350
VERB,30776,13.911817
NOUN,27991,12.652901
PRON,21115,9.544711
ADV,17767,8.031299
ADP,15952,7.210856
PART,13852,6.261583
ADJ,11407,5.156359
CCONJ,10433,4.716077
DET,6628,2.996085


#### Задача 7.

Посчитайте, какое соотношение токенов по частям речи имеет совпадающие со словоформой леммы (т.е., в скольких случаях токены с частью речи VERB, например, имели словарную форму: и сам токен, и лемма одинаковые). Что вы можете сказать о выбранных вами языках на основании этих данных? Ожидаются две таблички с процентами несовпадающих по лемме и токену слов для каждой части речи.

In [ ]:
def form_vs_lemma_stats(df):
  # словоформы могут иметь заглавные буквы, но решила не заменять исходные
  df['FORM_lower'] = df['FORM'].apply(lambda x: x.lower())

  diff_from_lemmas = df[df['FORM_lower'] != df['LEMMA']]
  dfl_counts = diff_from_lemmas[diff_from_lemmas['UPOS'] !='PUNCT']['UPOS'].value_counts()

  stats = pandas.DataFrame()
  stats.index = dfl_counts.index
  stats['count'] = dfl_counts.values
  stats['total'] = ''

  for pos in stats.index:
    total = df['UPOS'].value_counts()[pos]
    stats.at[pos,'total'] = total

  stats['percent'] = stats['count']/stats['total'] * 100

  return stats[['count','percent']]


In [ ]:
form_vs_lemma_stats(doc_ru)

,count,percent
VERB,26316,85.508188
NOUN,18413,65.781858
ADJ,9226,80.880161
PRON,8728,41.335543
PROPN,5572,98.636927
DET,4999,75.42245
AUX,1704,70.911361
ADP,847,5.309679
NUM,798,44.088398
ADV,785,4.418304


#### Задача 8.

Посчитайте медианную длину предложения для ваших текстов (медиана - это если взять все длины всех ваших предложений, упорядочить их от маленького к большому и выбрать то число, которое оказалось посередине, а если чисел четное количество, то взять среднее арифметическое двух чисел посередине. Например, если у вас пять предложений длинами 1, 2, 6, 7, 8, то медиана - 6, а если шесть предложений длинами 1, 1, 7, 9, 10, 11, то медиана - (7 + 9) / 2 = 8). Возьмите любые два предложения (одно русское и второе на другом языке) и постройте для них деревья зависимостей. Изучите связи зависимостей (deprel) и вершины: согласны ли вы с разбором?

In [ ]:
from statistics import median

In [ ]:
def sent_medium(text):
  sents = sent_tokenize(text)
  sent_lengths = [len(word_tokenize(sent)) for sent in sents]
  return median(sent_lengths)

In [ ]:
# это только если первое = читинг
def find_median(sent_lengths):
  count = len(sent_lengths)
  if count % 2 == 1:
    return sorted(sent_lengths)[count//2]
  else:
    return (sorted(sent_lengths)[count//2] + sorted(sent_lengths)[count//2 - 1]) /2

In [ ]:
sent_medium(text_de)

12

In [ ]:
sent_medium(text_ru)


11.0

In [ ]:
doc_ru = make_ud('ru','Он благополучно избегнул встречи с своею хозяйкой на лестнице.')

In [ ]:
doc_de = make_ud('de','Es gelang ihm, eine Begegnung mit seiner Wirtin auf der Treppe zu vermeiden.')

#### Задача 9.

Посчитайте частотные списки токенов для каждой категории связей зависимостей (т.е., нужно выделить все токены в тексте, которые получали, например, ярлык amod, и посчитать их частоты). Выведите по первые три самых частотных токена для каждой категории (punct можно не выводить).

In [ ]:
!pip install pyconll

Looking in indexes: https://pypi.org/simple, https://us-python.pkg.dev/colab-wheels/public/simple/


In [ ]:
import pyconll
from collections import Counter

text = pyconll.load_from_file('ru.conllu')
rel = set()
obj = {}
for sentence in text:
    for token in sentence:
        obj[token.deprel]=[]
print(obj)

{'ROOT': [], 'cc': [], 'conj': [], 'punct': [], 'amod': [], 'nmod': [], 'nsubj:pass': [], 'parataxis': [], 'obl': [], 'advmod': [], 'acl': [], 'case': [], 'det': [], 'nummod': [], 'nsubj': [], 'appos': [], 'obj': [], 'nummod:gov': [], 'fixed': [], 'iobj': [], 'mark': [], 'advcl': [], 'acl:relcl': [], 'cop': [], 'csubj': [], 'xcomp': [], 'ccomp': [], 'aux:pass': [], 'aux': [], 'orphan': [], 'discourse': [], 'flat:name': [], 'compound': [], 'expl': [], 'flat:foreign': [], 'csubj:pass': [], 'flat': []}


In [ ]:
for sentence in text:
    for token in sentence:
        obj[token.deprel].append(token.lemma)

print(obj)

for k in obj:
  c = Counter(obj[k]).most_common(3)
  print(k, c)

IOPub data rate exceeded.
The notebook server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--NotebookApp.iopub_data_rate_limit`.

Current values:
NotebookApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
NotebookApp.rate_limit_window=3.0 (secs)



#### Задача 10.

Некоторые предлоги в русском языке могут управлять разными падежами (например, "я еду в Лондон" vs "я живу в Лондоне"). Давайте проанализируем эти предлоги и их падежи. Необходимо:



*   составить список таких предлогов (РГ-80 вам в помощь)
*   взять достаточно большой текст (можно большое художественное произведение)
*   сделать морфоразбор этого текста (лучше не pymorphy)
*   Посчитать, как часто и какие падежи встречаются у слова, идущего после предлога.

Примечания: во-первых, имейте в виду, что иногда после предлога могут идти самые неожиданные вещи: "я что, должен ехать на, черт побери, северный полюс?". Во-вторых, неплохо бы учитывать отсутствие пунктуации (конечно, в норме, как нам кажется, предлог обязательно требует зависимое, но! "да иди ты на!") Эти штуки можно отсеять, если просто учитывать только заранее определенные падежи, а не считать все, какие встретились (так и None можно огрести).

In [ ]:
!pip install natasha

with open("dost.txt", "r") as dost:
  text = dost.read()

from natasha import (
    Segmenter,
    MorphVocab,
    NewsMorphTagger,
    Doc
)

segmenter = Segmenter()
morph_vocab = MorphVocab()
morph_tagger = NewsMorphTagger(emb)

doc = Doc(text)

doc.segment(segmenter)

doc.tag_morph(morph_tagger)

In [ ]:
"""вот предлоги, у которых бывает несколько падежей (согласно русграму)"""

prep = ["меж", "между", "промеж", "промежду", "за", "под", "подо", "в", "во", "на", "об", "о", "обо", "со", "с","по"]

In [ ]:
# словарь для хранения падежей после каждого предлога
case_counter = defaultdict(lambda: defaultdict(int))

# основная тусовка
for i in range(1, len(doc.tokens)):
  if doc.tokens[i-1].text.lower() in prep: #если предыдущее слово - нужный нам предлог, ака если текст нашего токена (смаленькойбуквы) есть в списке прелогов
    try: #попробуй получить падеж текущего слова
      case_info = doc.tokens[i].feats['Case']
      preposition = doc.tokens[i-1].text.lower() #запоминаем предыдущее слово-предлог
      case_counter[preposition][case_info] += 1 #записываем в наш двухэтажный словарь счетчик к конткретному предлогу в конкретный падеж прибавляем циферку
    except:
      pass #если у текущего слова нет падежа - забей. переходим к следующему токену


# вывод результатов
for preposition, cases in case_counter.items():
    print(f"\nПредлог: '{preposition}'")
    for case, count in cases.items():
        print(f"  {case}: {count}")